In [ ]:
# ============================================================
# RUNTIME SETTINGS
# ============================================================
# Required: CPU (Standard)
# GPU: Not required
# High-RAM: Recommended for large datasets
#
# SETUP: Add GITHUB_TOKEN to Colab Secrets (key icon in sidebar)
# ============================================================

import subprocess
from google.colab import userdata

# Get GitHub token from Colab Secrets (for private repo access)
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets.
"
        "1. Click the key icon in the left sidebar
"
        "2. Add a secret named 'GITHUB_TOKEN' with your GitHub token
"
        "3. Toggle 'Notebook access' ON"
    )

# Install package from private GitHub repo
repo_url = f"git+https://{token}@github.com/SilasPignotti/urban-tree-transfer.git"
subprocess.run(["pip", "install", repo_url, "-q"], check=True)

print("OK: Package installed")


In [ ]:
# Mount Google Drive for data files
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted")


In [ ]:
# Package imports
from urban_tree_transfer.config import PROJECT_CRS, RANDOM_SEED
from urban_tree_transfer.utils import ExecutionLog, save_figure, setup_plotting
from urban_tree_transfer.utils.plotting import PUBLICATION_STYLE

from pathlib import Path
from datetime import datetime, timezone
import json

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle, Rectangle

setup_plotting()
log = ExecutionLog("exp_06_proximity")

print("OK: Package imports complete")


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DRIVE_DIR = Path("/content/drive/MyDrive/dev/urban-tree-transfer")
INPUT_DIR = DRIVE_DIR / "data" / "phase_2_features"
OUTPUT_DIR = DRIVE_DIR / "outputs" / "phase_2"
METADATA_DIR = OUTPUT_DIR / "metadata"
LOGS_DIR = OUTPUT_DIR / "logs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "exp_06_proximity"

CITIES = ["berlin", "leipzig"]
THRESHOLDS_TO_TEST = [5, 10, 15, 20, 30]  # meters

for d in [METADATA_DIR, LOGS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Input (Phase 2):  {INPUT_DIR}")
print(f"Output (JSONs):   {METADATA_DIR}")
print(f"Figures:          {FIGURES_DIR}")
print(f"Logs (Drive):     {LOGS_DIR}")
print(f"Cities:           {CITIES}")
print(f"Random seed:      {RANDOM_SEED}")


In [ ]:
# ============================================================
# SECTION 1: Data Loading & Validation
# ============================================================

log.start_step("Data Loading")

required_columns = {"geometry", "genus_latin", "city"}
city_data = {}

total_trees = 0

for city in CITIES:
    path = INPUT_DIR / f"trees_clean_{city}.gpkg"
    print(f"Loading: {path}")
    gdf = gpd.read_file(path)

    # Ensure city column exists
    if "city" not in gdf.columns:
        gdf["city"] = city
        print(f"  Added missing city column for {city}.")

    missing = required_columns - set(gdf.columns)
    if missing:
        raise ValueError(f"Missing required columns for {city}: {missing}")

    # Validate CRS (EPSG:25833)
    if gdf.crs is None:
        raise ValueError(f"CRS missing for {city}. Expected {PROJECT_CRS}.")

    if gdf.crs.to_string() != PROJECT_CRS:
        print(f"  Reprojecting {city} to {PROJECT_CRS} (was {gdf.crs.to_string()})")
        gdf = gdf.to_crs(PROJECT_CRS)

    gdf = gdf[gdf["genus_latin"].notna()].copy()

    print(f"  {city}: {len(gdf):,} rows")
    city_data[city] = gdf
    total_trees += len(gdf)

log.end_step(status="success", records=total_trees)


In [ ]:
# ============================================================
# SECTION 2: Nearest Different-Genus Distance
# ============================================================

log.start_step("Nearest Neighbor Analysis")

np.random.seed(RANDOM_SEED)

def compute_nearest_diff_genus(gdf: gpd.GeoDataFrame, genus_col: str = "genus_latin") -> pd.Series:
    distances = pd.Series(index=gdf.index, dtype="float64")
    genera = sorted(gdf[genus_col].dropna().unique())

    for genus in genera:
        same_genus = gdf[gdf[genus_col] == genus]
        diff_genus = gdf[gdf[genus_col] != genus]

        if diff_genus.empty:
            distances.loc[same_genus.index] = np.inf
            continue

        diff_geom = diff_genus.geometry
        distances.loc[same_genus.index] = same_genus.geometry.apply(
            lambda geom: diff_geom.distance(geom).min()
        )

    return distances

for city, gdf in city_data.items():
    print(f"Computing nearest different-genus distance: {city}")
    city_data[city] = gdf.copy()
    city_data[city]["nearest_diff_genus_m"] = compute_nearest_diff_genus(gdf)

log.end_step(status="success")


In [ ]:
# ============================================================
# SECTION 3: Distance Distribution Analysis
# ============================================================

log.start_step("Distance Distribution")

stats_by_city = {}

for city, gdf in city_data.items():
    distances = gdf["nearest_diff_genus_m"].to_numpy()
    distances = distances[np.isfinite(distances)]

    if len(distances) == 0:
        raise ValueError(f"No finite distances for {city}. Check input data.")

    percentiles = np.percentile(distances, [10, 25, 50, 75, 90, 95, 99])
    stats_by_city[city] = {
        "count": int(len(distances)),
        "mean": float(np.mean(distances)),
        "std": float(np.std(distances)),
        "min": float(np.min(distances)),
        "max": float(np.max(distances)),
        "p10": float(percentiles[0]),
        "p25": float(percentiles[1]),
        "p50": float(percentiles[2]),
        "p75": float(percentiles[3]),
        "p90": float(percentiles[4]),
        "p95": float(percentiles[5]),
        "p99": float(percentiles[6]),
    }

# Visualization 1: Histogram per city
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
city_colors = {"berlin": "#1f77b4", "leipzig": "#ff7f0e"}

for idx, city in enumerate(CITIES):
    ax = axes[idx]
    distances = city_data[city]["nearest_diff_genus_m"]
    distances = distances[np.isfinite(distances)]

    sns.histplot(distances, bins=40, color=city_colors[city], ax=ax)
    for t in THRESHOLDS_TO_TEST:
        ax.axvline(t, color="black", linestyle="--", linewidth=1)

    ax.set_title(f"{city.title()}")
    ax.set_xlabel("Nearest different-genus distance (m)")
    ax.set_ylabel("Count")

fig.suptitle("Distribution of Nearest Different-Genus Distance", fontsize=14, fontweight="bold")
plt.tight_layout()
save_figure(fig, FIGURES_DIR / "distance_distribution.png")

log.end_step(status="success")


In [ ]:
# ============================================================
# SECTION 4: Threshold Sensitivity Analysis
# ============================================================

log.start_step("Threshold Sensitivity")

rows = []
for city, gdf in city_data.items():
    total = len(gdf)
    for threshold in THRESHOLDS_TO_TEST:
        removed = int((gdf["nearest_diff_genus_m"] < threshold).sum())
        kept = total - removed
        retention = kept / total if total else 0.0
        rows.append(
            {
                "city": city,
                "threshold": threshold,
                "trees_removed": removed,
                "trees_kept": kept,
                "retention_rate": retention,
            }
        )

sensitivity_results = pd.DataFrame(rows)

# Visualization 2: Retention Rate per Threshold
fig, ax = plt.subplots(figsize=PUBLICATION_STYLE["figsize"])

for city in CITIES:
    data = sensitivity_results[sensitivity_results["city"] == city]
    ax.plot(
        data["threshold"],
        data["retention_rate"],
        marker="o",
        label=city.title(),
    )

ax.axhline(0.85, color="red", linestyle="--", label="85% retention target")
ax.set_title("Retention Rate vs Proximity Threshold")
ax.set_xlabel("Threshold (m)")
ax.set_ylabel("Retention Rate")
ax.set_ylim(0, 1.0)
ax.legend(loc="best")

save_figure(fig, FIGURES_DIR / "retention_rate_sensitivity.png")

log.end_step(status="success")


In [ ]:
# ============================================================
# SECTION 5: Genus-Specific Impact
# ============================================================

log.start_step("Genus-Specific Impact")

pixel_size_m = 10
min_pixel_coverage = 2
min_threshold_for_pixels = pixel_size_m * min_pixel_coverage

threshold_evaluations = []

for threshold in THRESHOLDS_TO_TEST:
    evaluation = {"threshold": threshold, "per_city": {}}
    for city, gdf in city_data.items():
        gdf = gdf.copy()
        gdf["removed"] = gdf["nearest_diff_genus_m"] < threshold

        genus_stats = (
            gdf.groupby("genus_latin")["removed"]
            .agg(total_trees="count", removed="sum")
            .reset_index()
        )
        genus_stats["removal_rate"] = genus_stats["removed"] / genus_stats["total_trees"]

        max_deviation = float(genus_stats["removal_rate"].max() - genus_stats["removal_rate"].min())
        retention_rate = float(
            sensitivity_results[
                (sensitivity_results["city"] == city)
                & (sensitivity_results["threshold"] == threshold)
            ]["retention_rate"].iloc[0]
        )

        evaluation["per_city"][city] = {
            "retention_rate": retention_rate,
            "max_deviation": max_deviation,
            "genus_stats": genus_stats,
        }

    evaluation["passes_retention"] = all(
        evaluation["per_city"][city]["retention_rate"] >= 0.85 for city in CITIES
    )
    evaluation["passes_uniformity"] = all(
        evaluation["per_city"][city]["max_deviation"] < 0.10 for city in CITIES
    )
    evaluation["covers_two_pixels"] = threshold >= min_threshold_for_pixels

    threshold_evaluations.append(evaluation)

# Select best passing threshold: smallest threshold that meets all criteria
passing_thresholds = [
    e for e in threshold_evaluations
    if e["passes_retention"] and e["passes_uniformity"] and e["covers_two_pixels"]
]

if passing_thresholds:
    recommended = sorted(passing_thresholds, key=lambda x: x["threshold"])[0]
    recommended_threshold = recommended["threshold"]
else:
    # Fallback: choose threshold with highest average retention
    avg_retention = []
    for e in threshold_evaluations:
        avg = np.mean([e["per_city"][c]["retention_rate"] for c in CITIES])
        avg_retention.append((avg, e["threshold"]))
    recommended_threshold = max(avg_retention)[1]

print(f"Recommended threshold: {recommended_threshold}m")

# Compute genus impact for recommended threshold
impact_rows = []
for city, gdf in city_data.items():
    gdf = gdf.copy()
    gdf["removed"] = gdf["nearest_diff_genus_m"] < recommended_threshold
    genus_stats = (
        gdf.groupby("genus_latin")["removed"]
        .agg(total_trees="count", removed="sum")
        .reset_index()
    )
    genus_stats["removal_rate"] = genus_stats["removed"] / genus_stats["total_trees"]
    genus_stats["city"] = city
    impact_rows.append(genus_stats)

genus_impact = pd.concat(impact_rows, ignore_index=True)

# Visualization 3: Removal Rate per Genus (Grouped Bar Chart)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for idx, city in enumerate(CITIES):
    ax = axes[idx]
    subset = genus_impact[genus_impact["city"] == city].sort_values("removal_rate")
    ax.bar(subset["genus_latin"], subset["removal_rate"], color=city_colors[city])
    ax.axhline(subset["removal_rate"].mean(), color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{city.title()}")
    ax.set_xlabel("Genus")
    ax.set_ylabel("Removal rate")
    ax.tick_params(axis="x", rotation=90)

fig.suptitle(
    f"Genus-Specific Removal Rate ({recommended_threshold}m threshold)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
save_figure(fig, FIGURES_DIR / "genus_specific_impact.png")

max_deviation = genus_impact.groupby("city")["removal_rate"].apply(lambda x: x.max() - x.min())
is_uniform = max_deviation < 0.10

log.end_step(status="success")


In [ ]:
# ============================================================
# SECTION 6: Spatial Distribution
# ============================================================

log.start_step("Spatial Distribution")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, city in enumerate(CITIES):
    ax = axes[idx]
    gdf = city_data[city].copy()
    gdf["affected"] = gdf["nearest_diff_genus_m"] < recommended_threshold

    gdf.plot(ax=ax, color="lightgray", markersize=2)
    gdf[gdf["affected"]].plot(ax=ax, color="red", markersize=6, label="Affected")

    ax.set_title(f"{city.title()}")
    ax.set_axis_off()

fig.suptitle(
    f"Spatial Distribution of Affected Trees ({recommended_threshold}m threshold)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
fig_name = f"spatial_distribution_{recommended_threshold}m.png"
save_figure(fig, FIGURES_DIR / fig_name)

log.end_step(status="success")


In [ ]:
# ============================================================
# SECTION 7: Sentinel-2 Pixel Contamination
# ============================================================

log.start_step("Pixel Contamination Analysis")

fig, ax = plt.subplots(figsize=(8, 8))

# Draw a 10m x 10m pixel
pixel = Rectangle((-5, -5), 10, 10, fill=False, edgecolor="black", linewidth=2)
ax.add_patch(pixel)

# Draw tree at center
ax.scatter([0], [0], color="green", s=80, label="Tree")

# Threshold circles
thresholds = [5, 10, 20]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for t, c in zip(thresholds, colors):
    ax.add_patch(Circle((0, 0), radius=t, fill=False, edgecolor=c, linewidth=2))
    ax.text(t + 0.5, 0, f"{t}m", color=c, va="center")

ax.set_title("Sentinel-2 Pixel Contamination Zones", fontsize=14, fontweight="bold")
ax.set_xlim(-25, 25)
ax.set_ylim(-25, 25)
ax.set_aspect("equal")
ax.set_xlabel("Meters")
ax.set_ylabel("Meters")

save_figure(fig, FIGURES_DIR / "pixel_contamination_schematic.png")

log.end_step(status="success")


In [ ]:
# ============================================================
# SECTION 8: Threshold Recommendation
# ============================================================

log.start_step("Threshold Recommendation")

recommended_eval = next(
    (e for e in threshold_evaluations if e["threshold"] == recommended_threshold),
    None,
)

print("THRESHOLD RECOMMENDATION")
print("=" * 60)
print(f"Recommended: {recommended_threshold}m")

for city in CITIES:
    retention = recommended_eval["per_city"][city]["retention_rate"]
    deviation = recommended_eval["per_city"][city]["max_deviation"]
    print(f"Retention ({city.title()}): {retention * 100:.1f}%")
    print(f"Genus deviation ({city.title()}): {deviation * 100:.2f}%")

print(f"Pixel coverage: {recommended_threshold / 10:.1f} pixels")
print(f"Meets retention target: {recommended_eval['passes_retention']}")
print(f"Genus uniformity: {recommended_eval['passes_uniformity']}")
print(f"Covers >=2 pixels: {recommended_eval['covers_two_pixels']}")

log.end_step(status="success")


In [ ]:
# ============================================================
# SECTION 9: Export JSON Configuration
# ============================================================

log.start_step("Export Configuration")

impact_per_threshold = {}
for threshold in THRESHOLDS_TO_TEST:
    impact_per_threshold[str(threshold)] = {}
    for city in CITIES:
        row = sensitivity_results[
            (sensitivity_results["city"] == city)
            & (sensitivity_results["threshold"] == threshold)
        ].iloc[0]
        impact_per_threshold[str(threshold)][city] = {
            "trees_removed": int(row["trees_removed"]),
            "retention_rate": float(row["retention_rate"]),
        }

output = {
    "version": "1.0",
    "created": datetime.now(timezone.utc).isoformat(),
    "recommended_threshold_m": int(recommended_threshold),
    "justification": (
        f"Threshold {recommended_threshold}m provides the best passing balance between "
        "spectral purity and retention, meeting retention and genus-uniformity criteria "
        "while covering at least two Sentinel-2 pixels."
    ),
    "thresholds_tested": THRESHOLDS_TO_TEST,
    "impact_per_threshold": impact_per_threshold,
    "genus_specific_uniform": bool(is_uniform.all()),
    "max_genus_deviation_percent": float(max_deviation.max() * 100),
    "distance_percentiles": stats_by_city,
    "validation": {
        "retention_exceeds_85_percent": bool(recommended_eval["passes_retention"]),
        "genus_impact_uniform": bool(recommended_eval["passes_uniformity"]),
        "covers_two_pixels": bool(recommended_eval["covers_two_pixels"]),
    },
}

json_path = METADATA_DIR / "proximity_filter.json"
json_path.write_text(json.dumps(output, indent=2), encoding="utf-8")
print(f"Saved: {json_path}")

log.end_step(status="success")


In [ ]:
# ============================================================
# SUMMARY & MANUAL SYNC INSTRUCTIONS
# ============================================================

# Save execution log
log.summary()
log_path = LOGS_DIR / f"{log.notebook}_execution.json"
log.save(log_path)
print(f"Execution log saved: {log_path}")

print("
" + "=" * 60)
print("OUTPUT SUMMARY")
print("=" * 60)

print("
--- JSON CONFIGURATIONS ---")
json_files = list(METADATA_DIR.glob("*.json"))
for f in sorted(json_files):
    print(f"  {f.name}")

print("
--- PLOTS CREATED ---")
plot_files = list(FIGURES_DIR.glob("*.png"))
for f in sorted(plot_files):
    print(f"  {f.name}")

print(f"
Total plots: {len(plot_files)}")

print("
" + "=" * 60)
print("⚠️  MANUAL SYNC REQUIRED")
print("=" * 60)
print("
JSON configurations must be synced to Git repo:")
print("1. Download from Google Drive:")
for f in json_files:
    print(f"   - {f.relative_to(DRIVE_DIR)}")

print("
2. Copy to local repo:")
print("   - Destination: outputs/phase_2/metadata/")

print("
3. Commit and push to Git")
print("   - git add outputs/phase_2/metadata/*.json")
print("   - git commit -m 'Add proximity filter config'")
print("   - git push")

print("
4. (Optional) Commit plots for documentation:")
print(f"   - Source: {FIGURES_DIR}")
print("   - Destination: outputs/phase_2/figures/exp_06_proximity/")

print("
" + "=" * 60)
print("NOTEBOOK COMPLETE")
print("=" * 60)
